In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import List

@dataclass
class NewsItem:
    id: str
    source: str
    title: str
    body: str = ""
    published_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    relevant: float = field(default=1.0)


class NewsSource(ABC):
    """Abstract base class for all news sources"""

    @abstractmethod
    def fetch(self) -> List[NewsItem]:
        """Fetch news items from this source"""
        pass

    @property
    @abstractmethod
    def source_name(self) -> str:
        """Name of the news source"""
        pass

In [ ]:
import pandas as pd

import csv
from datetime import datetime, timezone
from typing import List

def load_news_from_csv(csv_filename: str) -> List[NewsItem]:
    news_items = []
    with open(csv_filename, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            published_at = row.get("published_at")
            try:
                published_dt = datetime.fromisoformat(published_at) if published_at else datetime.now(timezone.utc)
            except Exception:
                published_dt = datetime.now(timezone.utc)

            news_item = NewsItem(
                id=row.get("id", ""),
                source=row.get("source", ""),
                title=row.get("title", ""),
                body=row.get("body", ""),
                published_at=published_dt,
                relevant=float(row.get("relevant", 1.0))
            )
            news_items.append(news_item)
    return news_items


data = load_news_from_csv("training_data.csv")

df_finetune = pd.DataFrame(data)

# Combine title and body into one text field
df_finetune["text"] = df_finetune["title"] + " " + df_finetune["body"]


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import mean_squared_error, mean_absolute_error

MODEL_PATH = "./it_news_classifier"
model_name = "distilbert-base-uncased"

# Prepare dataset
df_finetune = df_finetune[["text", "relevant"]].rename(columns={"relevant": "label"})
hf_dataset = Dataset.from_pandas(df_finetune)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH if os.path.exists(MODEL_PATH) else model_name)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_dataset = hf_dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

# Load existing model or create new
if os.path.exists(MODEL_PATH):
    print("✅ Loading existing model...")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
else:
    print("🚀 Training new model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )

    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        preds = preds.squeeze()
        mse = mean_squared_error(labels, preds)
        mae = mean_absolute_error(labels, preds)
        return {"mse": mse, "mae": mae}

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=0.01,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=100,
        weight_decay=0.01,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset['train'],
        eval_dataset=tokenized_dataset['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(MODEL_PATH)
    tokenizer.save_pretrained(MODEL_PATH)



    import matplotlib.pyplot as plt

    # Extract logged metrics
    logs = trainer.state.log_history

    train_loss = [log["loss"] for log in logs if "loss" in log]
    eval_loss = [log["eval_loss"] for log in logs if "eval_loss" in log]
    mse = [log["eval_mse"] for log in logs if "eval_mse" in log]
    mae = [log["eval_mae"] for log in logs if "eval_mae" in log]
    eval_steps = list(range(1, len(eval_loss) + 1))

    # Plot training/eval loss
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_loss, label="Training Loss")
    plt.plot(eval_steps, eval_loss, label="Eval Loss")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Loss")
    plt.title("Training & Evaluation Loss")
    plt.legend()

    # Plot regression metrics
    plt.subplot(1, 2, 2)
    plt.plot(eval_steps, mse, label="MSE")
    plt.plot(eval_steps, mae, label="MAE")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Regression Metrics")
    plt.legend()

    plt.tight_layout()
    plt.show()


    plt.savefig("training_metrics.png")

#api key: 76c4c5adf2e67e4b22cd9a982cf744fdd8e68a71


Map:   0%|          | 0/9937 [00:00<?, ? examples/s]

🚀 Training new model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-10-3505622470.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Mse,Mae
1,No log,0.246494,0.246494,0.490487
2,10.881900,0.249273,0.249273,0.499234
3,0.244700,0.246946,0.246946,0.489193
4,0.245000,0.246919,0.246919,0.489255
5,0.244800,0.247237,0.247237,0.488581
6,0.245300,0.246992,0.246992,0.489088
7,0.245000,0.246269,0.246269,0.491738
8,0.245400,0.247170,0.247170,0.488714
9,0.244500,0.246772,0.246772,0.495316
10,0.245400,0.246875,0.246875,0.489360


Epoch,Training Loss,Validation Loss,Mse,Mae
1,No log,0.246494,0.246494,0.490487
2,10.881900,0.249273,0.249273,0.499234
3,0.244700,0.246946,0.246946,0.489193
4,0.245000,0.246919,0.246919,0.489255
5,0.244800,0.247237,0.247237,0.488581
6,0.245300,0.246992,0.246992,0.489088
7,0.245000,0.246269,0.246269,0.491738
8,0.245400,0.247170,0.247170,0.488714
9,0.244500,0.246772,0.246772,0.495316
10,0.245400,0.246875,0.246875,0.489360
